# Phase 1: Foundations — From RNNs to Transformers

**Goal:** Understand *why* each architecture exists by seeing what problem it solves and where the previous one falls short.

**How to use this notebook:**
- Run each code cell and observe the output
- Fill in the `YOUR UNDERSTANDING` sections in your own words
- These notes become your interview prep material later

In [ ]:
import torch
import torch.nn as nn
import numpy as np
import matplotlib.pyplot as plt

print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
print(f"MPS available: {torch.backends.mps.is_available()}")

---
## 1. The Problem: Why Sequence Order Matters

Regular neural networks (MLPs) treat input as a flat vector — they have no concept of **order**.
But language is inherently sequential: *"dog bites man"* means something very different from *"man bites dog"*.

Let's see this problem firsthand.

In [ ]:
# A simple MLP has no concept of word order
# If we represent words as one-hot vectors and sum them, order is lost

vocab = {"the": 0, "dog": 1, "bites": 2, "man": 3}

def bag_of_words(sentence, vocab):
    """Bag-of-words throws away word order entirely."""
    vec = torch.zeros(len(vocab))
    for word in sentence.split():
        if word in vocab:
            vec[vocab[word]] += 1
    return vec

s1 = bag_of_words("the dog bites the man", vocab)
s2 = bag_of_words("the man bites the dog", vocab)

print(f"'the dog bites the man' -> {s1}")
print(f"'the man bites the dog' -> {s2}")
print(f"\nIdentical representations? {torch.equal(s1, s2)}")
print("\n^ This is the problem. Two very different meanings, same representation.")

### YOUR UNDERSTANDING

*Why can't a simple feed-forward neural network handle language well? Write your answer here:*

> (your notes here)


---
## 2. RNNs: The First Solution to Sequence Modeling

RNNs process tokens **one at a time**, carrying a hidden state forward.
Think of it as reading a sentence word-by-word and keeping a running "summary" in your head.

**Key idea:** `h_t = f(h_{t-1}, x_t)` — the hidden state at time `t` depends on the previous hidden state AND the current input.

In [ ]:
# Let's build a minimal RNN from scratch to see exactly what happens

class SimpleRNN(nn.Module):
    def __init__(self, input_size, hidden_size):
        super().__init__()
        self.hidden_size = hidden_size
        self.W_xh = nn.Linear(input_size, hidden_size, bias=False)
        self.W_hh = nn.Linear(hidden_size, hidden_size, bias=False)
        self.tanh = nn.Tanh()
    
    def forward(self, x_sequence):
        """Process a sequence one token at a time."""
        h = torch.zeros(1, self.hidden_size)
        hidden_states = []
        
        for t, x_t in enumerate(x_sequence):
            h = self.tanh(self.W_xh(x_t) + self.W_hh(h))
            hidden_states.append(h.detach().clone())
            print(f"  Step {t}: processed token, hidden state norm = {h.norm():.4f}")
        
        return h, hidden_states

# Simulate a 5-token sequence with embedding dim = 4
rnn = SimpleRNN(input_size=4, hidden_size=8)
fake_tokens = [torch.randn(1, 4) for _ in range(5)]

print("Processing 5 tokens through RNN:")
final_h, all_h = rnn(fake_tokens)
print(f"\nFinal hidden state shape: {final_h.shape}")
print("The final hidden state is the RNN's 'summary' of the entire sequence.")

### RNN Strengths
- Processes tokens in order — captures sequence information
- Shared weights across all time steps — efficient parameter usage

### But there's a fatal flaw... let's see it.

In [ ]:
# THE VANISHING GRADIENT PROBLEM
# As sequences get longer, early tokens have less and less influence
# on the final hidden state. Information "fades away."

def measure_gradient_flow(seq_length, hidden_size=16):
    """Measure how much the first token affects the final output."""
    rnn = nn.RNN(input_size=4, hidden_size=hidden_size, batch_first=True)
    
    x = torch.randn(1, seq_length, 4, requires_grad=True)
    output, h_n = rnn(x)
    
    loss = h_n.sum()
    loss.backward()
    
    grad_at_first_token = x.grad[0, 0, :].norm().item()
    grad_at_last_token = x.grad[0, -1, :].norm().item()
    
    return grad_at_first_token, grad_at_last_token

print("Gradient magnitude reaching first vs last token:")
print(f"{'Seq Length':<15} {'First Token':<15} {'Last Token':<15} {'Ratio':<10}")
print("-" * 55)

for length in [5, 10, 25, 50, 100, 200]:
    first, last = measure_gradient_flow(length)
    ratio = first / last if last > 0 else 0
    print(f"{length:<15} {first:<15.6f} {last:<15.6f} {ratio:<10.4f}")

print("\n^ Notice how the gradient reaching the first token shrinks")
print("  as the sequence gets longer. The RNN 'forgets' early context.")

### YOUR UNDERSTANDING

*In your own words, what is the vanishing gradient problem and why does it matter for language?*

> (your notes here)

*Think of a real example where forgetting early context would give a wrong answer:*

> (your example here)


---
## 3. The Encoder-Decoder Architecture

For tasks like translation or summarization, we need to read an **entire input** and then produce an **output sequence**.

The encoder-decoder pattern:
1. **Encoder** reads the input sequence → produces a context vector
2. **Decoder** takes the context vector → generates the output sequence

**The bottleneck problem:** The entire input gets compressed into a single fixed-size vector.
Imagine summarizing a 500-word paragraph into a single number — you'd lose information.

In [ ]:
# Visualize the bottleneck: same hidden size must encode sequences of any length

hidden_size = 64  # fixed context vector size

fig, ax = plt.subplots(1, 1, figsize=(10, 4))

seq_lengths = [5, 10, 20, 50, 100, 200, 500]
info_per_dim = [length / hidden_size for length in seq_lengths]

bars = ax.bar([str(s) for s in seq_lengths], info_per_dim, color='steelblue', edgecolor='black')
ax.set_xlabel('Input Sequence Length (tokens)')
ax.set_ylabel(f'Tokens per Hidden Dimension (hidden_size={hidden_size})')
ax.set_title('The Bottleneck Problem: More tokens squeezed into the same vector')
ax.axhline(y=1, color='red', linestyle='--', label='1:1 ratio')
ax.legend()

for bar, val in zip(bars, info_per_dim):
    ax.text(bar.get_x() + bar.get_width()/2., bar.get_height() + 0.05,
            f'{val:.1f}x', ha='center', va='bottom', fontsize=9)

plt.tight_layout()
plt.show()

print("When ratio > 1, each hidden dimension must encode more than one token's worth of info.")
print("This is why longer sequences degrade quality in vanilla encoder-decoder models.")

---
## 4. Attention: The Breakthrough Idea

**Core insight:** Instead of compressing everything into one vector, let the decoder **look back** at all encoder hidden states and decide which ones are relevant for each output step.

The analogy: Instead of memorizing an entire textbook, you get to flip back and re-read the relevant page for each exam question.

### How Attention Works (step by step)
1. **Query:** What the decoder is currently looking for
2. **Keys:** What each encoder position "advertises" about itself
3. **Values:** The actual information at each encoder position
4. **Score:** Query • Key → how relevant is this position?
5. **Weights:** Softmax(scores) → normalized attention distribution
6. **Output:** Weighted sum of Values

In [ ]:
# Let's implement attention from scratch and visualize it

def scaled_dot_product_attention(query, key, value):
    """
    The fundamental attention operation.
    query: (seq_len_q, d_k)
    key:   (seq_len_k, d_k)
    value: (seq_len_k, d_v)
    """
    d_k = query.shape[-1]
    
    # Step 1: Compute similarity scores
    scores = torch.matmul(query, key.transpose(-2, -1)) / np.sqrt(d_k)
    
    # Step 2: Normalize to get attention weights
    weights = torch.softmax(scores, dim=-1)
    
    # Step 3: Weighted sum of values
    output = torch.matmul(weights, value)
    
    return output, weights


# Simulate: 4 encoder positions, decoder asking 1 query
d_k = 8
encoder_states = torch.randn(4, d_k)  # 4 positions
decoder_query = torch.randn(1, d_k)   # 1 query

output, weights = scaled_dot_product_attention(
    query=decoder_query,
    key=encoder_states,
    value=encoder_states
)

tokens = ["The", "cat", "sat", "down"]
print("Attention weights (how much the decoder 'looks at' each input token):")
for token, w in zip(tokens, weights[0]):
    bar = "█" * int(w.item() * 40)
    print(f"  {token:>6}: {w.item():.4f} {bar}")

print(f"\nWeights sum to: {weights.sum().item():.4f} (always 1.0 due to softmax)")

In [ ]:
# Visualize attention as a heatmap — this is how attention is typically shown in papers

# Simulate a translation scenario with multiple decoder steps
source_tokens = ["The", "cat", "is", "sleeping", "."]
target_tokens = ["Le", "chat", "dort", "."]

# Fake but illustrative attention pattern
# In reality these come from training — here we craft them to show the concept
attention_matrix = torch.tensor([
    [0.55, 0.10, 0.05, 0.05, 0.25],  # "Le" attends mostly to "The"
    [0.05, 0.70, 0.05, 0.10, 0.10],  # "chat" attends mostly to "cat"
    [0.02, 0.05, 0.20, 0.68, 0.05],  # "dort" attends mostly to "sleeping" (+ "is")
    [0.05, 0.05, 0.05, 0.05, 0.80],  # "." attends to "."
])

fig, ax = plt.subplots(figsize=(7, 5))
im = ax.imshow(attention_matrix.numpy(), cmap='Blues', aspect='auto')

ax.set_xticks(range(len(source_tokens)))
ax.set_xticklabels(source_tokens, fontsize=12)
ax.set_yticks(range(len(target_tokens)))
ax.set_yticklabels(target_tokens, fontsize=12)
ax.set_xlabel('Source (English)', fontsize=12)
ax.set_ylabel('Target (French)', fontsize=12)
ax.set_title('Attention Heatmap: Which source words does each target word attend to?', fontsize=11)

for i in range(len(target_tokens)):
    for j in range(len(source_tokens)):
        val = attention_matrix[i, j].item()
        color = 'white' if val > 0.5 else 'black'
        ax.text(j, i, f'{val:.2f}', ha='center', va='center', color=color, fontsize=10)

plt.colorbar(im, ax=ax, label='Attention Weight')
plt.tight_layout()
plt.show()

print('Key insight: The decoder can "look back" at the relevant source word at each step.')
print('No more bottleneck — the full encoder sequence stays accessible.')

### YOUR UNDERSTANDING

*How does attention solve the two RNN problems (vanishing gradients + bottleneck)?*

> (your notes here)

*In the heatmap above, why does "dort" attend to both "is" and "sleeping"?*

> (your notes here)


---
## 5. Self-Attention: Attending to Yourself

The attention above was **cross-attention** (decoder looks at encoder).

**Self-attention** is the same mechanism but within a single sequence — each token looks at every other token in the same sentence to understand context.

This is the core building block of Transformers.

In [ ]:
# Self-attention: every token attends to every other token in the same sequence

sentence = ["The", "bank", "of", "the", "river"]
seq_len = len(sentence)
d_model = 16

# Random embeddings (in a real model these are learned)
embeddings = torch.randn(seq_len, d_model)

# Learned projection matrices
W_Q = nn.Linear(d_model, d_model, bias=False)
W_K = nn.Linear(d_model, d_model, bias=False)
W_V = nn.Linear(d_model, d_model, bias=False)

Q = W_Q(embeddings)  # queries from the same sequence
K = W_K(embeddings)  # keys from the same sequence
V = W_V(embeddings)  # values from the same sequence

output, self_attn_weights = scaled_dot_product_attention(Q, K, V)

# Visualize self-attention
fig, ax = plt.subplots(figsize=(6, 5))
im = ax.imshow(self_attn_weights.detach().numpy(), cmap='Purples', aspect='auto')

ax.set_xticks(range(seq_len))
ax.set_xticklabels(sentence, fontsize=11)
ax.set_yticks(range(seq_len))
ax.set_yticklabels(sentence, fontsize=11)
ax.set_xlabel('Attending TO', fontsize=11)
ax.set_ylabel('Attending FROM', fontsize=11)
ax.set_title('Self-Attention: Each token attends to all tokens (including itself)', fontsize=10)

for i in range(seq_len):
    for j in range(seq_len):
        val = self_attn_weights[i, j].item()
        color = 'white' if val > 0.4 else 'black'
        ax.text(j, i, f'{val:.2f}', ha='center', va='center', color=color, fontsize=9)

plt.colorbar(im, ax=ax)
plt.tight_layout()
plt.show()

print('Notice: "bank" can attend to "river" to understand it means riverbank, not a financial bank.')
print('This contextual understanding is impossible for bag-of-words or vanilla RNNs.')

---
## 6. The Transformer: Putting It All Together

The Transformer (Vaswani et al., 2017 — "Attention Is All You Need") removes RNNs entirely:

- **No recurrence** → processes all tokens in parallel → much faster training
- **Self-attention** → every token sees every other token → no vanishing gradient
- **Positional encoding** → since there's no recurrence, position must be injected explicitly
- **Multi-head attention** → multiple attention "perspectives" in parallel

### Architecture at a Glance
```
Input → Embeddings + Positional Encoding → [N × (Self-Attention → Feed-Forward)] → Output
```

In [ ]:
# Positional Encoding — why it's needed and how it works

def positional_encoding(seq_len, d_model):
    """
    Sinusoidal positional encoding from 'Attention Is All You Need'.
    Uses sin for even dimensions, cos for odd dimensions,
    with wavelengths forming a geometric progression.
    """
    pe = torch.zeros(seq_len, d_model)
    position = torch.arange(0, seq_len, dtype=torch.float).unsqueeze(1)
    div_term = torch.exp(torch.arange(0, d_model, 2).float() * -(np.log(10000.0) / d_model))
    
    pe[:, 0::2] = torch.sin(position * div_term)
    pe[:, 1::2] = torch.cos(position * div_term)
    
    return pe

pe = positional_encoding(seq_len=50, d_model=64)

fig, axes = plt.subplots(1, 2, figsize=(14, 4))

# Heatmap of full encoding
im = axes[0].imshow(pe.numpy().T, cmap='RdBu', aspect='auto')
axes[0].set_xlabel('Position in sequence')
axes[0].set_ylabel('Encoding dimension')
axes[0].set_title('Positional Encoding (each position gets a unique pattern)')
plt.colorbar(im, ax=axes[0])

# Show a few individual dimensions
for dim in [0, 1, 4, 5, 10, 11]:
    axes[1].plot(pe[:, dim].numpy(), label=f'dim {dim}', alpha=0.7)
axes[1].set_xlabel('Position in sequence')
axes[1].set_ylabel('Encoding value')
axes[1].set_title('Individual dimensions: different frequencies encode different scales')
axes[1].legend(fontsize=8)

plt.tight_layout()
plt.show()

print("Low dimensions = low frequency (capture long-range position patterns)")
print("High dimensions = high frequency (capture fine-grained position differences)")
print("\nThis gives the model a unique 'fingerprint' for each position.")

In [ ]:
# Multi-Head Attention: multiple "perspectives" on the same data

# Instead of one large attention, split into multiple heads
# Each head can learn different relationships (syntax, semantics, coreference, etc.)

d_model = 64
num_heads = 4

multihead_attn = nn.MultiheadAttention(embed_dim=d_model, num_heads=num_heads, batch_first=True)

# Fake sequence: batch=1, seq_len=6, d_model=64
x = torch.randn(1, 6, d_model)
attn_output, attn_weights = multihead_attn(x, x, x)  # self-attention: Q=K=V=x

print(f"Input shape:           {x.shape}")
print(f"Output shape:          {attn_output.shape}  (same as input!)")
print(f"Attention weights:     {attn_weights.shape}  (averaged across {num_heads} heads)")
print(f"\nEach head uses d_k = d_model / num_heads = {d_model} / {num_heads} = {d_model // num_heads}")
print(f"All {num_heads} heads run in parallel, then concatenate and project.")

In [ ]:
# The key advantage: Transformers process ALL tokens in parallel
# Let's compare computation patterns

import time

def time_rnn(seq_len, d_model=64, hidden_size=64):
    rnn = nn.RNN(input_size=d_model, hidden_size=hidden_size, batch_first=True)
    x = torch.randn(1, seq_len, d_model)
    start = time.perf_counter()
    for _ in range(100):
        _ = rnn(x)
    return (time.perf_counter() - start) / 100

def time_transformer(seq_len, d_model=64, nhead=4):
    layer = nn.TransformerEncoderLayer(d_model=d_model, nhead=nhead, batch_first=True)
    x = torch.randn(1, seq_len, d_model)
    start = time.perf_counter()
    for _ in range(100):
        _ = layer(x)
    return (time.perf_counter() - start) / 100

seq_lengths = [10, 25, 50, 100, 200]
rnn_times = [time_rnn(s) for s in seq_lengths]
tf_times = [time_transformer(s) for s in seq_lengths]

fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(seq_lengths, rnn_times, 'o-', label='RNN', color='coral', linewidth=2)
ax.plot(seq_lengths, tf_times, 's-', label='Transformer Layer', color='steelblue', linewidth=2)
ax.set_xlabel('Sequence Length')
ax.set_ylabel('Time per forward pass (seconds)')
ax.set_title('RNN vs Transformer: Forward Pass Speed')
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print("RNNs scale linearly (sequential processing).")
print("Transformers scale quadratically in memory but are parallelizable on GPU.")
print("In practice, Transformers train MUCH faster on modern hardware.")

---
## 7. Three Transformer Architectures for LLMs

The original Transformer has both encoder and decoder. But different LLM families use different parts:

| Architecture | What it does | Examples | Best for |
|---|---|---|---|
| **Encoder-only** | Bidirectional: sees full context both ways | BERT, RoBERTa | Classification, NER, sentiment |
| **Decoder-only** | Autoregressive: generates left-to-right | GPT-2, GPT-4, LLaMA | Text generation, chatbots |
| **Encoder-Decoder** | Full seq-to-seq | T5, FLAN-T5, BART | Translation, summarization |

### The key difference is in the **attention mask**:

In [ ]:
# Visualize the difference: bidirectional vs causal (masked) attention

seq_len = 6
tokens = ["I", "love", "large", "language", "models", "!"]

# Encoder (bidirectional): every token sees every other token
encoder_mask = torch.ones(seq_len, seq_len)

# Decoder (causal): each token can only see previous tokens (and itself)
decoder_mask = torch.tril(torch.ones(seq_len, seq_len))

fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))

for ax, mask, title in [
    (axes[0], encoder_mask, 'Encoder (Bidirectional)\nBERT, RoBERTa'),
    (axes[1], decoder_mask, 'Decoder (Causal/Masked)\nGPT-2, GPT-4, LLaMA')
]:
    im = ax.imshow(mask.numpy(), cmap='Greens', aspect='auto', vmin=0, vmax=1)
    ax.set_xticks(range(seq_len))
    ax.set_xticklabels(tokens, fontsize=10)
    ax.set_yticks(range(seq_len))
    ax.set_yticklabels(tokens, fontsize=10)
    ax.set_title(title, fontsize=11)
    ax.set_xlabel('Can attend TO')
    ax.set_ylabel('Token')
    
    for i in range(seq_len):
        for j in range(seq_len):
            val = mask[i, j].item()
            symbol = '✓' if val > 0 else '✗'
            color = 'white' if val > 0.5 else 'red'
            ax.text(j, i, symbol, ha='center', va='center', color=color, fontsize=12)

plt.tight_layout()
plt.show()

print('Encoder: "models" can see "I" → good for understanding full context')
print('Decoder: "models" can only see ["I", "love", "large", "language", "models"] → needed for generation')
print('\nWhy causal masking? During generation, future tokens don\'t exist yet!')

---
## 8. Summary: The Evolution

```
Feed-Forward Networks
  │  Problem: No sequence awareness
  ▼
RNNs / LSTMs
  │  Problem: Sequential processing (slow), vanishing gradients (forgets early context)
  ▼
Attention Mechanism
  │  Solution: Direct connections to all positions, weighted by relevance
  ▼
Transformers ("Attention Is All You Need")
  │  Key wins: Parallel processing, no vanishing gradients, scales beautifully
  ▼
Large Language Models (GPT, BERT, T5, LLaMA, ...)
     Transformers trained on massive data → emergent capabilities
```

### YOUR UNDERSTANDING — Final Reflection

*Summarize the journey from RNNs to Transformers in 3-4 sentences, as if explaining to a colleague:*

> (your notes here)

*Why are decoder-only models (GPT family) dominant for generative AI, while encoder-only models (BERT) are better for classification?*

> (your notes here)

*What's one thing that surprised you or clicked during this notebook?*

> (your notes here)


In [ ]:
RNN alone
  │  Problem: vanishing gradients
  ▼
RNN-based Encoder-Decoder (Seq2Seq)
  │  Solved: can now do input → output (translation, summarization)
  │  New problem: bottleneck (entire input squeezed into one vector)
  ▼
Attention added to Encoder-Decoder
  │  Solved: both vanishing gradients AND bottleneck
  ▼
Transformer (Attention Is All You Need)
     Removed RNNs entirely, attention does everything

---
## Next Up: Phase 1b — Tokenizers & Embeddings

In the next notebook, we'll get hands-on with:
- Different tokenizer types (BPE, WordPiece, SentencePiece)
- How embeddings represent meaning
- Visualizing embedding spaces

**Before moving on**, make sure you've filled in all the `YOUR UNDERSTANDING` sections above!